# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the FAIR² dataset (logistic regression results on rangeland management practices in Northern Kenya) using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

We’ll explore available record sets, load data using entity `@id`s, and perform some common exploratory and preprocessing steps.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets (`@id`), their fields (`@id`), and the first example record from each.

In [ ]:
# List all available record sets and their fields using @id

import pprint

if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = []

if not record_sets:
    print("No record sets found in the metadata. Attempting to read by inferring from dataset ...")
    # As per Croissant 1.0 spec, recordSet is usually a property; fallback to scanning the dataset
    record_sets = dataset.record_sets

# Display summary of record sets and their fields
record_set_ids = []
for rs in dataset.record_sets:
    print(f"Record set: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            if isinstance(field, dict) and '@id' in field:
                print(f"    {field['@id']}")
            else:
                print(f"    {field}")
    print()
    # Show the first record for each record set
    try:
        rec_gen = dataset.records(record_set=rs['@id'])
        example = next(rec_gen)
        print(f"  First record snippet: {str(dict(list(example.items())[:5])) if isinstance(example, dict) else example}\n")
    except Exception as e:
        print(f"  Could not read records for {rs['@id']}: {e}\n")
if not record_set_ids:
    print("No record sets detected in this dataset.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for exploration.

Below, replace `<record_set_id>` with the actual `@id` of the record set(s) you wish to analyze.

In [ ]:
# Load records from desired record sets

# Example: Suppose one record set is '@id': 'https://sen.science/recordset/results' (replace as needed below)
# We'll automatically take all record set IDs found above.

dfs = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dfs[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set {record_set_id}, shape: {dfs[record_set_id].shape}")
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Let's preview columns of the largest record set (or first if only one)
if dfs:
    # Pick the largest DataFrame
    main_rs_id = max(dfs, key=lambda k: dfs[k].shape[0])
    print(f"Columns in DataFrame for record set {main_rs_id}:")
    print(dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic numeric filtering, normalization, and grouping using `@id`-referenced fields from your DataFrame.

In [ ]:
# Example: Analyze a numeric field in the main record set
# List numeric columns
if dfs:
    df = dfs[main_rs_id]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields (@id): {numeric_cols}")

    if numeric_cols:
        numeric_field = numeric_cols[0]  # Pick first numeric field for illustration
        print(f"Analyzing field (likely @id): {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a non-numeric column if available
        group_fields = [col for col in df.columns if col not in numeric_cols]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field} (first 5):")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("Data not loaded for EDA.")

## 5. Visualization
Here we plot the distribution of a selected numeric field and visualize grouped means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_cols:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped, barplot of group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant packages using the `mlcroissant` library,
- Explore available record sets and their schema using `@id` references,
- Extract data, perform simple EDA and preprocessing, and,
- Visualize distributions and grouped trends using pandas and seaborn.

All entity and field references were by their Croissant `@id` for maximal traceability and reproducibility.

_Explore further: consult the Croissant schema (linked above) for more fields, types, and relationships in this dataset!_